# Лабораторная работа №5: Рекомендательные системы


In [1]:
import importlib.util
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from surprise import Dataset

spec = importlib.util.spec_from_file_location("slim_impl", "src/SLIM.py")
slim_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(slim_mod)
MySLIM = slim_mod.SLIM

from SLIM import SLIM as SLIMRef, SLIMatrix

print("Все импорты успешны")

Все импорты успешны


## 1. Загрузка и исследование датасета

In [2]:
data = Dataset.load_builtin("ml-100k")
df = pd.DataFrame(data.raw_ratings, columns=["user_id", "item_id", "rating", "timestamp"])
df["rating"] = df["rating"].astype(float)

nu = df["user_id"].nunique()
ni = df["item_id"].nunique()
print(f"Пользователей: {nu}")
print(f"Фильмов:       {ni}")
print(f"Оценок:        {len(df)}")
print(f"Разреженность: {1 - len(df)/(nu*ni):.2%}")
print(f"Средняя оценка: {df.rating.mean():.2f}")
df.head()

Пользователей: 943
Фильмов:       1682
Оценок:        100000
Разреженность: 93.70%
Средняя оценка: 3.53


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


## 2. Предобработка

In [3]:
user_ids = sorted(df["user_id"].unique())
item_ids = sorted(df["item_id"].unique())
user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {v: i for i, v in enumerate(item_ids)}

df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["item_id"].map(item2idx)

n_users = len(user_ids)
n_items = len(item_ids)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

R_train = np.zeros((n_users, n_items))
for _, row in train_df.iterrows():
    R_train[int(row["user_idx"]), int(row["item_idx"])] = row["rating"]

R_test = np.zeros((n_users, n_items))
for _, row in test_df.iterrows():
    R_test[int(row["user_idx"]), int(row["item_idx"])] = row["rating"]

test_users   = test_df["user_idx"].values.astype(int)
test_items   = test_df["item_idx"].values.astype(int)
test_ratings = test_df["rating"].values

print(f"Матрица оценок: {R_train.shape}")

Train: 80000, Test: 20000
Матрица оценок: (943, 1682)


## 3. Своя реализация SLIM



In [13]:
slim = MySLIM(alpha=0.00212, l1_ratio=0.5, max_iter=50, tol=1e-7)


t0 = time.time()
slim.fit(R_train)
our_time = time.time() - t0

print(f"Время обучения: {our_time:.1f} сек ({our_time/60:.1f} мин)")

SLIM training: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1682/1682 [00:00<00:00, 2310.03it/s]

Время обучения: 0.8 сек (0.0 мин)


In [14]:
R_pred_ours = slim.predict_all(R_train)

ours_preds = np.clip(R_pred_ours[test_users, test_items], 1, 5)
ours_rmse  = np.sqrt(np.mean((ours_preds - test_ratings) ** 2))
spW = np.mean(slim.W == 0)

print(f"Наш SLIM RMSE:           {ours_rmse:.4f}")
print(f"Разреженность матрицы W: {spW:.2%}")

Наш SLIM RMSE:           2.3393
Разреженность матрицы W: 99.50%


## 4. Эталон: KarypisLab/SLIM



In [15]:
R_sparse = csr_matrix(R_train)
trainmat  = SLIMatrix(R_sparse)

params   = {"algo": "cd", "nthreads": 4, "l1r": 1.0, "l2r": 1.0}
slim_ref = SLIMRef()

t0 = time.time()
slim_ref.train(params, trainmat)
ref_time = time.time() - t0

print(f"Время обучения: {ref_time:.1f} сек")

Learning takes 0.958 secs.
Время обучения: 1.0 сек


In [16]:
W_ref      = slim_ref.to_csr()
R_pred_ref = (R_sparse @ W_ref).toarray()

ref_preds = np.clip(R_pred_ref[test_users, test_items], 1, 5)
ref_rmse  = np.sqrt(np.mean((ref_preds - test_ratings) ** 2))

print(f"KarypisLab SLIM RMSE: {ref_rmse:.4f}")

KarypisLab SLIM RMSE: 2.4530


## 5. NDCG@10


In [17]:
def dcg_at_k(relevance, k):
    rel = np.array(relevance[:k], dtype=float)
    if len(rel) == 0:
        return 0.0
    return np.sum(rel / np.log2(np.arange(2, len(rel) + 2)))

def ndcg_at_k(actual, predicted, k=10):
    pred_order  = np.argsort(-predicted)
    ideal_order = np.argsort(-actual)
    dcg  = dcg_at_k(actual[pred_order],  k)
    idcg = dcg_at_k(actual[ideal_order], k)
    return dcg / idcg if idcg > 0 else 0.0

def mean_ndcg(R_true, R_pred, k=10):
    scores = []
    for u in range(R_true.shape[0]):
        rated = R_true[u] > 0
        if rated.sum() < 2:
            continue
        actual    = R_true[u, rated]
        predicted = R_pred[u, rated]
        scores.append(ndcg_at_k(actual, predicted, k=min(k, len(actual))))
    return float(np.mean(scores))

In [22]:
ours_ndcg = mean_ndcg(R_test, R_pred_ours, k=10)
ref_ndcg  = mean_ndcg(R_test, R_pred_ref,  k=10)

print(f"Наш SLIM NDCG@10:        {ours_ndcg:.4f}")
print(f"KarypisLab SLIM NDCG@10: {ref_ndcg:.4f}")

Наш SLIM NDCG@10:        0.9009
KarypisLab SLIM NDCG@10: 0.9017


## 6. Сводные результаты

In [10]:
results = pd.DataFrame({
    "Модель":           ["SLIM (своя реализация)", "SLIM (KarypisLab, эталон)"],
    "RMSE":             [f"{ours_rmse:.4f}",        f"{ref_rmse:.4f}"],
    "NDCG@10":          [f"{ours_ndcg:.4f}",        f"{ref_ndcg:.4f}"],
    "Время обучения":   [f"{our_time:.1f} сек",     f"{ref_time:.1f} сек"],
})
print(results.to_string(index=False))

                   Модель   RMSE NDCG@10 Время обучения
   SLIM (своя реализация) 2.3393  0.9009        0.9 сек
SLIM (KarypisLab, эталон) 2.4530  0.9017        1.0 сек


## 7. Латентная факторная модель (LFM / Funk SVD)

In [8]:
spec_lfm = importlib.util.spec_from_file_location("lfm_impl", "src/LFM.py")
lfm_mod  = importlib.util.module_from_spec(spec_lfm)
spec_lfm.loader.exec_module(lfm_mod)
MyLFM = lfm_mod.LFM

lfm = MyLFM(n_factors=50, n_epochs=20, lr=0.005, reg_u=0.02, reg_i=0.02)

t0 = time.time()
lfm.fit(R_train)
lfm_time = time.time() - t0

print(f"Время обучения LFM: {lfm_time:.1f} сек")

Время обучения LFM: 13.0 сек


In [9]:
R_pred_lfm = lfm.predict_all()

lfm_preds = np.clip(R_pred_lfm[test_users, test_items], 1, 5)
lfm_rmse  = np.sqrt(np.mean((lfm_preds - test_ratings) ** 2))

print(f"LFM (своя реализация) RMSE: {lfm_rmse:.4f}")

LFM (своя реализация) RMSE: 0.9329


### 7.1 Эталон: surprise.SVD

In [18]:
from surprise import SVD, Reader
from surprise import Dataset as SurpriseDataset

reader = Reader(rating_scale=(1, 5))
surprise_train_data = SurpriseDataset.load_from_df(
    train_df[["user_id", "item_id", "rating"]], reader
).build_full_trainset()

svd_ref = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
t0 = time.time()
svd_ref.fit(surprise_train_data)
svd_ref_time = time.time() - t0

svd_ref_preds = np.array([
    svd_ref.predict(str(uid), str(iid)).est
    for uid, iid in zip(test_df["user_id"], test_df["item_id"])
])
svd_ref_preds = np.clip(svd_ref_preds, 1, 5)
svd_ref_rmse = np.sqrt(np.mean((svd_ref_preds - test_ratings) ** 2))

print(f"surprise.SVD RMSE:           {svd_ref_rmse:.4f}")
print(f"Время обучения surprise.SVD: {svd_ref_time:.1f} сек")

surprise.SVD RMSE:           0.9313
Время обучения surprise.SVD: 0.3 сек


### 7.2 NDCG@10 для LFM

In [19]:
# Build full prediction matrix for surprise.SVD via internal factor matrices (fast, no loops)
trainset = surprise_train_data

def safe_inner_uid(uid):
    try: return trainset.to_inner_uid(uid)
    except ValueError: return 0

def safe_inner_iid(iid):
    try: return trainset.to_inner_iid(iid)
    except ValueError: return 0

inner_uids = np.array([safe_inner_uid(u) for u in user_ids])
inner_iids = np.array([safe_inner_iid(i) for i in item_ids])

R_pred_svd_ref = (
    trainset.global_mean
    + svd_ref.bu[inner_uids, None]
    + svd_ref.bi[inner_iids, None].T
    + svd_ref.pu[inner_uids] @ svd_ref.qi[inner_iids].T
)

lfm_ndcg     = mean_ndcg(R_test, R_pred_lfm,     k=10)
svd_ref_ndcg = mean_ndcg(R_test, R_pred_svd_ref, k=10)

print(f"LFM (своя реализация) NDCG@10: {lfm_ndcg:.4f}")
print(f"surprise.SVD NDCG@10:          {svd_ref_ndcg:.4f}")

LFM (своя реализация) NDCG@10: 0.9171
surprise.SVD NDCG@10:          0.9166


## 8. Итоговое сравнение всех моделей

In [23]:
all_results = pd.DataFrame({
    "Модель": [
        "SLIM (своя реализация)",
        "SLIM (KarypisLab, эталон)",
        "LFM/Funk SVD (своя реализация)",
        "surprise.SVD (эталон)",
    ],
    "RMSE": [
        f"{ours_rmse:.4f}",
        f"{ref_rmse:.4f}",
        f"{lfm_rmse:.4f}",
        f"{svd_ref_rmse:.4f}",
    ],
    "NDCG@10": [
        f"{ours_ndcg:.4f}",
        f"{ref_ndcg:.4f}",
        f"{lfm_ndcg:.4f}",
        f"{svd_ref_ndcg:.4f}",
    ],
    "Время обучения": [
        f"{our_time:.1f} сек",
        f"{ref_time:.1f} сек",
        f"{lfm_time:.1f} сек",
        f"{svd_ref_time:.1f} сек",
    ],
})
print(all_results.to_string(index=False))

                        Модель   RMSE NDCG@10 Время обучения
        SLIM (своя реализация) 2.3393  0.9009        0.8 сек
     SLIM (KarypisLab, эталон) 2.4530  0.9017        1.0 сек
LFM/Funk SVD (своя реализация) 0.9329  0.9171       13.0 сек
         surprise.SVD (эталон) 0.9313  0.9166        0.3 сек
